# Тест кнопки СТОП для Stable Diffusion 1.5

Сначала нажмите Run All. После появления двух кнопок нажмите «ЗАПУСТИТЬ КОШКУ». Во время генерации можно нажать «СТОП». Остановка проверяется между шагами генерации; после остановки или завершения модель удаляется из памяти GPU.

**Важно:** кнопка СТОП прерывает генерацию и освобождает VRAM, но сама по себе не выключает всю GPU-сессию Kaggle.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors

In [ ]:
import gc, threading, time, torch, ipywidgets as widgets
from IPython.display import display
from diffusers import StableDiffusionPipeline

stop_event = threading.Event()
worker = None
status = widgets.HTML(value='<b>Статус:</b> готово к запуску')
start_button = widgets.Button(description='ЗАПУСТИТЬ КОШКУ', button_style='success', layout=widgets.Layout(width='220px', height='48px'))
stop_button = widgets.Button(description='СТОП', button_style='danger', disabled=True, layout=widgets.Layout(width='220px', height='48px'))
output = widgets.Output()

class StopRequested(Exception):
    pass

def stop_generation(_):
    stop_event.set()
    status.value = '<b>Статус:</b> получена команда СТОП. Останавливаю на ближайшем шаге...'
    stop_button.disabled = True

def generate_cat():
    pipe = None
    try:
        if not torch.cuda.is_available():
            raise RuntimeError('GPU CUDA не обнаружена. Включите GPU в настройках Kaggle.')
        stop_event.clear()
        status.value = '<b>Статус:</b> загружаю модель на GPU...'
        with output:
            print('GPU:', torch.cuda.get_device_name(0))
        pipe = StableDiffusionPipeline.from_pretrained(
            'stable-diffusion-v1-5/stable-diffusion-v1-5',
            torch_dtype=torch.float16, variant='fp16', use_safetensors=True,
            safety_checker=None, requires_safety_checker=False
        ).to('cuda:0')
        pipe.enable_attention_slicing()
        def check_stop(pipeline, step_index, timestep, callback_kwargs):
            status.value = f'<b>Статус:</b> генерация, шаг {step_index + 1} из 24'
            if stop_event.is_set():
                raise StopRequested('Генерация остановлена кнопкой СТОП.')
            return callback_kwargs
        with output:
            print('Начинаю рисовать кота. Кнопку СТОП можно нажать в любой момент.')
        result = pipe(
            'A fluffy ginger cat sitting on a windowsill, warm morning sunlight, realistic detailed fur, cozy room, professional photography',
            height=512, width=512, num_inference_steps=24, guidance_scale=7.0,
            callback_on_step_end=check_stop
        )
        result.images[0].save('/kaggle/working/stop-button-cat.png')
        status.value = '<b>Статус:</b> готово! Кот сохранён в /kaggle/working/stop-button-cat.png'
        with output:
            print('Картинка готова: /kaggle/working/stop-button-cat.png')
    except StopRequested as e:
        status.value = '<b>Статус:</b> генерация остановлена. Освобождаю GPU-память.'
        with output:
            print(str(e))
    except Exception as e:
        status.value = '<b>Статус:</b> ошибка. Подробности ниже.'
        with output:
            print(type(e).__name__ + ':', e)
    finally:
        if pipe is not None:
            del pipe
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            with output:
                print('Память CUDA после очистки: allocated =', round(torch.cuda.memory_allocated()/1024**2), 'MiB; reserved =', round(torch.cuda.memory_reserved()/1024**2), 'MiB')
        start_button.disabled = False
        stop_button.disabled = True

def start_generation(_):
    global worker
    if worker is not None and worker.is_alive():
        return
    stop_event.clear()
    start_button.disabled = True
    stop_button.disabled = False
    worker = threading.Thread(target=generate_cat, daemon=True)
    worker.start()

start_button.on_click(start_generation)
stop_button.on_click(stop_generation)
display(widgets.VBox([status, widgets.HBox([start_button, stop_button]), output]))
print('Панель готова. Запустите кошку зелёной кнопкой, затем нажмите красную СТОП.')